<img src='images/logo_cryptis.png' style='width: 1200px; float:center'>


# Sécurité et implémentations sur cartes à puce
### ANNEE 2026-2027

------------------------------------------

<img src='./images/DIY.jpg' style='width: 100px'>


# U TO DO IT !

---------
#  Basics on Simple Side-Channel Analysis and first reverse side-channel analysis

--------

* Perform Simple Power Analysis on one and/or several traces (SPA) to recognize the cryptographic algorithm in the traces
* Explain how you recognize it 
* Explain what kind of operations you can identify and where in the traces
* Is the trace containing the whole algorithm computation ? Yes ? No ? All or how many round?
* Apply signal processing such as vertical mean, variance on several traces to remove noise and have a better trace to observe
* Use REVERSE side-channel advanced analysis such as correlation or NICV or ANOVA with the proper selection functions to detect leakage or not for each of the AES main operations: KeyAddition, Shiftrows, SuBbytes, Mixcolumn

----------------
### import

In [1]:
import warnings
import estraces
import numpy as np
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

In [2]:
import plotly.graph_objects as go;
from plotly_resampler import FigureResampler

In [ ]:
ths = estraces.read_ths_from_ets_file("../SideSCA-Traces-Public/traces_stm32_set.ets") 
print(ths)

### Plot and observe traces

<img src='./images/DIY.jpg' style='width: 100px'>


### How to manipulate metadata

In [ ]:
#plaintext 
ths.plaintext

In [ ]:
# 0 to 5 first plaintexts
ths.plaintext[0:5]

In [ ]:
# Same with other metadata
ths.key[0:5]

In [ ]:
ths.ciphertext[0:5]

In [ ]:
#Manipulate the traces
ths.samples[0:5]

In [9]:
#Length of the traces
len(ths.samples[0])

8000


### Verify the AES computation is correct inside the metadata

<img src='./images/DIY.jpg' style='width: 100px'>


## Observe the trace 

Open the trace in full screen, zoom and observe

* What do you see?
* What kind of information do you visualize?
* highlight the rounds and the operations


<img src='./images/DIY.jpg' style='width: 100px'>

---------
## Using signal processing can be very useful to provide better readability
- Applying vertical mean, variance (for instance) on several traces allow to remove noise of the trace and allow identification of operations and sometimes even secrets 
- If only a single trace is available, then (horizontal) moving mean, moving variance or moving kurtosis can lead also to better data analysis and operations and secrets observation 


Compute the mean and the standard deviation on the set of traces

<img src='./images/DIY.jpg' style='width: 100px'>

---------
## Using an open source SCA tool to analyse the trace with side-channel reverse engineering

### Select your model

In [10]:
import scared

In [11]:
model_selected=scared.HammingWeight()
#model=scared.Value()
#model=scared.Monobit()    # and set the bit value

### Select your distinguisher

In [12]:
Distinguisher = scared.CPAReverse
#Distinguisher = scared.SNRReverse
#Distinguisher = scared.ANOVAReverse
#Distinguisher = scared.NICVReverse

In [13]:
container = scared.Container(ths)

In [14]:
@scared.reverse_selection_function
def identity_plaintext(plaintext):
    return plaintext

@scared.reverse_selection_function
def identity_ciphertext(ciphertext):
    return ciphertext

In [15]:
reverse_plaintext = Distinguisher(selection_function=identity_plaintext, model = model_selected)
reverse_ciphertext = Distinguisher(selection_function=identity_ciphertext, model = model_selected)

reverse_plaintext.run(container)
reverse_ciphertext.run(container)

In [ ]:
plt.rcParams['figure.figsize']=(20,4)
plt.subplot(121)
plt.grid(True)
plt.plot(reverse_plaintext.results.T)
plt.title('Plaintext Reverse')
plt.subplot(122)
plt.grid(True)
plt.plot(reverse_ciphertext.results.T)
plt.title('Ciphertext Reverse')
plt.show()

----
### Reverse on intermediate values - First round

In [17]:
def aes_function_firstRde_AddKey(plaintext, key):
    return scared.aes.encrypt(plaintext = plaintext, key = key, at_round=0, after_step=scared.aes.base.Steps.ADD_ROUND_KEY)

def aes_function_firstRde_SBout(plaintext, key):
    return scared.aes.encrypt(plaintext = plaintext, key = key, at_round=1, after_step=scared.aes.base.Steps.SUB_BYTES)

def aes_function_firstRde_ShiftRow(plaintext, key):
    return scared.aes.encrypt(plaintext = plaintext, key = key, at_round=1, after_step=scared.aes.base.Steps.SHIFT_ROWS)

def aes_function_firstRde_MixColumn(plaintext, key):
    return scared.aes.encrypt(plaintext = plaintext, key = key, at_round=1, after_step=scared.aes.base.Steps.MIX_COLUMNS)

In [18]:
container = scared.Container(ths)

In [19]:
Sel_AdK = scared.reverse_selection_function(aes_function_firstRde_AddKey)
Sel_SB = scared.reverse_selection_function(aes_function_firstRde_SBout)
Sel_SR = scared.reverse_selection_function(aes_function_firstRde_ShiftRow)
Sel_MXC = scared.reverse_selection_function(aes_function_firstRde_MixColumn)

In [20]:
Reverse_AdK = Distinguisher(selection_function = Sel_AdK, model = model_selected)
Reverse_SB = Distinguisher(selection_function = Sel_SB, model = model_selected)
Reverse_SR = Distinguisher(selection_function = Sel_SR, model = model_selected)
Reverse_MXC = Distinguisher(selection_function = Sel_MXC, model = model_selected)

In [21]:
Reverse_AdK.run(container)
Reverse_SB.run(container)
Reverse_SR.run(container)
Reverse_MXC.run(container)

In [ ]:
Reverse_SB.results.shape

### Plot results

In [ ]:
plt.rcParams['figure.figsize']=(20,8)

plt.subplot(221)
plt.grid(True)
plt.plot(Reverse_AdK.results.T)
plt.title('Reverse AddKey')
plt.subplot(222)
plt.grid(True)
plt.plot(Reverse_SB.results.T)
plt.title('Reverse SBox')
plt.subplot(223)
plt.grid(True)
plt.plot(Reverse_SR.results.T)
plt.title('Reverse ShitRow')
plt.subplot(224)
plt.grid(True)
plt.plot(Reverse_MXC.results.T)
plt.title('Reverse MixColumn')

plt.show()

# Conclusion

...


